In [ ]:
!pip install --upgrade transformers accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. 加载分词器
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# 2. 加载模型
# 关键修改：删除了 trust_remote_code=True，使用 Transformers 原生支持
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    # trust_remote_code=True,  <-- 这一行被注释掉了，不要加它
)

# 3. 创建流水线
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

# 4. 生成文本
prompt = "who is faker?"
result = generator(prompt)
print("生成结果：")
print(result[0]['generated_text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


生成结果：


Chatbot: I'm here to assist you with your inquiries. If you're referring to a person, I'm unable to provide personal opinions or judgments. However, I can help you with information on


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("正在重新加载模型和分词器，请稍候...")

# 1. 加载分词器 (定义 tokenizer)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# 2. 加载模型 (定义 model)
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    # trust_remote_code=True, # 如果你的 transformers 库是最新的，这行可以注释掉
)

print("加载完成！现在可以运行下一段代码了。")
prompt="The capital of France is"
input_ids=tokenizer(prompt,return_tensors="pt").input_ids
input_ids=input_ids.to("cuda")
model_output=model.model(input_ids)
lm_head_output=model.lm_head(model_output[0])
token_id=lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)
model_output[0].shape
lm_head_output.shape

正在重新加载模型和分词器，请稍候...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

加载完成！现在可以运行下一段代码了。


torch.Size([1, 5, 32064])

In [ ]:
import time
import torch

# 确保你已经加载了 model 和 tokenizer
# 如果没有，请先运行之前的加载代码

print("1. 正在准备提示词...")
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# ---------------------------------------------------------
# 测试 1: 启用缓存 (use_cache=True) - 默认模式
# ---------------------------------------------------------
print("\n开始测试：启用缓存 (use_cache=True)...")
start_time = time.time() # 开始计时

# 生成文本
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=True
)

end_time = time.time() # 结束计时
print(f"✅ 启用缓存耗时: {end_time - start_time:.4f} 秒")


# ---------------------------------------------------------
# 测试 2: 禁用缓存 (use_cache=False)
# ---------------------------------------------------------
print("\n开始测试：禁用缓存 (use_cache=False)...")
start_time = time.time() # 开始计时

# 生成文本
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=100,
    use_cache=False
)

end_time = time.time() # 结束计时
print(f"❌ 禁用缓存耗时: {end_time - start_time:.4f} 秒")

print("\n结论：你可以看到禁用缓存后，生成速度明显变慢了。")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1. 正在准备提示词...

开始测试：启用缓存 (use_cache=True)...
✅ 启用缓存耗时: 8.1610 秒

开始测试：禁用缓存 (use_cache=False)...
❌ 禁用缓存耗时: 30.6701 秒

结论：你可以看到禁用缓存后，生成速度明显变慢了。
